# Task 2: Build your first AI agent

In this notebook, you learn how to build AI agents using the Strands Agents framework. You start by creating a conversational agent, then add tools to make it more capable. You finally build a practical recipe assistant that can search the web for cooking information. You use Amazon Bedrock with the Amazon Nova Lite model to power your agents.

AI agents differ from traditional LLMs because they can take actions, use tools, and work toward goals autonomously. Rather than only responding to questions, agents can actually do things for you.

#### Scenario
You work for AnyCompany, a growing technology company that wants to explore how AI agents can help automate everyday tasks. Your team is interested in learning how these agents could help with tasks like research and customer support. You are starting with the basics of building intelligent agents.

## Task 2.1: Environment setup

In this task, you set up your environment by installing the necessary packages to get started with building AI agents.

In [ ]:
# Install the Strands Agents framework and tools
# %pip install strands-agents strands-agents-tools

## Task 2.2: Create your first AI agent

In this task, you create an AI agent that can have conversations. This agent uses Amazon Bedrock with the Amazon Nova Lite model to understand and respond to your messages. The system prompt defines the desired behavior of your agent.

In [ ]:
import warnings
warnings.filterwarnings(action="ignore", message=r"datetime.datetime.utcnow") 

from strands import Agent

# Create your first AI agent
agent = Agent(callback_handler=None,
    model="amazon.nova-lite-v1:0",
    system_prompt="You are a helpful assistant that provides concise responses."
)

Next, you test your agent by sending it a message.

<i aria-hidden="true" class="fas fa-sticky-note" style="color:#563377"></i> **Note:** The Strands framework uses Amazon Bedrock with the Amazon Nova Lite model by default. This model supports conversational interactions and can be enhanced with tools.

In [ ]:
# Send a message to the agent
response = agent("Hello! Tell me a joke.")
print(response)

## Task 2.3: Add tools to your agent

In this task, you give your agent tools so it can do more than chat.

- You add a calculator tool which is provided by the Strands Agents SDK.

- You create a weather tool using the @tool decorator. The tool takes a **city name** (for example `Seattle` or `Berlin`) and fetches the *real* current weather from the free [Open-Meteo](https://open-meteo.com/) API — no API key required.

<i aria-hidden="true" class="fas fa-sticky-note" style="color:#563377"></i> **Note:** Open-Meteo works with latitude/longitude, not city names. The tool therefore uses a small lookup table (`CITY_COORDINATES`) to map the supported cities to their coordinates before calling the API. If the agent asks for a city that is not in the table, the tool returns a message listing the supported cities so the agent can tell the user.

In [2]:
import json
import urllib.parse
import urllib.request

from strands import Agent, tool
from strands_tools import calculator

# Open-Meteo needs latitude/longitude, so we map the supported city names
# to their coordinates. Add more cities here if you like.
CITY_COORDINATES = {
    "seattle": (47.6062, -122.3321),
    "berlin": (52.5200, 13.4050),
    "new york": (40.7128, -74.0060),
    "london": (51.5074, -0.1278),
    "paris": (48.8566, 2.3522),
    "tokyo": (35.6762, 139.6503),
    "sydney": (-33.8688, 151.2093),
    "frankfurt": (50.1109, 8.6821),
    "munich": (48.1351, 11.5820),
}

# WMO weather interpretation codes returned by Open-Meteo -> human-readable text
WMO_CODES = {
    0: "clear sky", 1: "mainly clear", 2: "partly cloudy", 3: "overcast",
    45: "fog", 48: "depositing rime fog",
    51: "light drizzle", 53: "moderate drizzle", 55: "dense drizzle",
    61: "slight rain", 63: "moderate rain", 65: "heavy rain",
    71: "slight snow", 73: "moderate snow", 75: "heavy snow",
    80: "slight rain showers", 81: "moderate rain showers", 82: "violent rain showers",
    95: "thunderstorm", 96: "thunderstorm with slight hail", 99: "thunderstorm with heavy hail",
}


# Create a weather tool
@tool
def weather(city: str) -> str:
    """Get the current weather for a city.

    Args:
        city (str): Name of the city, e.g. "Seattle" or "Berlin".

    Returns:
        A short description of the current weather (temperature in Celsius,
        humidity, wind speed and conditions), or an error message if the city
        is not supported or the weather service is unavailable.
    """
    key = city.strip().lower()
    if key not in CITY_COORDINATES:
        supported = ", ".join(name.title() for name in CITY_COORDINATES)
        return f"Unknown city '{city}'. Supported cities: {supported}."

    latitude, longitude = CITY_COORDINATES[key]
    params = urllib.parse.urlencode({
        "latitude": latitude,
        "longitude": longitude,
        "current": "temperature_2m,relative_humidity_2m,wind_speed_10m,weather_code",
        "timezone": "auto",
    })
    url = f"https://api.open-meteo.com/v1/forecast?{params}"

    try:
        with urllib.request.urlopen(url, timeout=10) as resp:
            data = json.load(resp)
    except Exception as e:
        return f"Weather service error: {e}"

    current = data["current"]
    conditions = WMO_CODES.get(current["weather_code"], "unknown conditions")
    return (
        f"Current weather in {city.strip().title()}: {conditions}, "
        f"{current['temperature_2m']} degrees Celsius, "
        f"{current['relative_humidity_2m']}% humidity, "
        f"wind {current['wind_speed_10m']} km/h "
        f"(observed {current['time']} local time)."
    )


# Create an agent with tools
agent_with_tools = Agent(callback_handler=None,
    model="amazon.nova-lite-v1:0",
    tools=[calculator, weather],
    system_prompt=(
        "You are a helpful assistant. You can do math calculations and use the "
        "weather tool to tell the current weather for a city. "
        "RULE: You must never do arithmetic yourself, not even simple unit conversions. "
        "Every calculation, including Celsius to Fahrenheit, MUST be done by calling "
        "the calculator tool. Do not write formulas; call the tool."
    )
)

# Test the agent with both tools in one query
# Note: This query requires both tools - the weather tool to get temperature in Celsius,
# and the calculator tool to convert from Celsius to Fahrenheit.
# The agent will automatically determine which tools to use and in what order.
response = agent_with_tools("What is the weather in Seattle in Fahrenheit?")
print(response)

# The tool requires a city name, so the agent has to extract it from the question.
response = agent_with_tools("And how is the weather in Berlin right now?")
print(response)

DEPRECATION WARNING: calculator is deprecated. This warning becomes an error log in v0.9.0. To achieve similar functionality, use the shell tool vended by strands-agents (from strands.vended_tools import shell). This does change the security boundary: calculator only ever evaluated an expression checked against an AST allowlist, while shell executes arbitrary commands, so review it against your threat model before switching.


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ (3.9 * 9/5) + 32    │                                                                            │
│  │ Result    │ 39.02               │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

The current weather in Seattle is clear sky, with a temperature of 39.02 degrees Fahrenheit, 80% humidity, and wind at 1.8 km/h as of 2026-10-24 at 06:00 local time.

The current weather in Berlin is slight rain, with a temperature of 10.3 degrees Celsius, 70% humidity, and wind at 18.6 km/h as of 2026-10-24 at 15:00 local time.



Now explore how the agent analyze the question and decide to use only the calculator tool

In [ ]:
# Test with a math question that only needs the calculator tool
# The agent will analyze the question and decide to use only the calculator tool
math_query = "What is 25 * 4 + 18?"
print("=== Agent Chooses Calculator Tool ===")
print(f"Query: {math_query}")
response = agent_with_tools(math_query)
print(f"Response: {response}")

### Direct Tool Invocation

You can also call tools directly without going through the agent conversation. This is useful for testing or when you want to use a specific tool function.

**Specification for direct tool invocation:**
- Use `agent.tool.tool_name()` to call a specific tool directly
- Pass the required parameters as function arguments
- This bypasses the agent's natural language processing and tool selection logic
- Useful for programmatic access to tool functionality

In [ ]:
# Call the calculator tool directly
# Note: Direct tool invocation bypasses the agent's conversation flow.
# Use agent_with_tools.tool.calculator() to call the calculator tool directly
# without the agent deciding which tool to use. This is useful for testing
# specific tools or when you know exactly which tool function you need.
result = agent_with_tools.tool.calculator(expression="2 + 3 * 4")
print(f"Calculator result: {result}")

## Task 2.4: Configure logging

In this task, you set up logging to understand what your agent is doing behind the scenes. This helps you understand how the agent processes requests and uses tools.

The Strands framework uses Python's standard logging module to provide visibility into agent operations. You can configure different log levels to get more or less detail about what your agent is doing.

In [ ]:
import logging
from strands import Agent
import os

# Enable detailed logging to understand what the agent is doing
logging.getLogger("strands").setLevel(logging.INFO)

# Set up logging to write to both console and file
logging.basicConfig(
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    level=logging.INFO,
    handlers=[
        logging.StreamHandler()  # Console output
    ]
)

# Create a logger
logger = logging.getLogger("agent_activity")

# Create an agent with logging enabled
logger.info("Creating new agent with Nova Lite model")
logged_agent = Agent(callback_handler=None,model="amazon.nova-lite-v1:0")

logger.info("Sending message to agent: 'Hello! How are you?'")
response = logged_agent("Hello! How are you?")
print(response)

## Task 2.5: Explore model configuration

In this task, you learn how to configure different models and settings for your agents. You can specify which AI model to use and adjust its behavior with parameters like temperature. Temperature controls how creative or consistent the responses are. Lower temperature values make responses more consistent and predictable, while higher values make them more creative and varied.

In [ ]:
from strands import Agent
from strands.models import BedrockModel

# Create a custom model configuration
custom_model = BedrockModel(
    model_id="amazon.nova-lite-v1:0",
    temperature=0.3  # Lower temperature = more consistent responses
)

# Create an agent with the custom model
custom_agent = Agent(callback_handler=None,model=custom_model)
print("Agent created successfully!")

## Task 2.6: Build a recipe assistant agent

In this task, you create a more practical agent that can help with cooking. This recipe assistant can search the web for recipes and cooking information, showing how agents can be useful in real-world scenarios.

First, install the web search package that your recipe agent needs.

In [ ]:
%pip install ddgs

Create a web search tool that your recipe agent can use to find cooking information and recipes online.

In [ ]:
from strands import Agent, tool
from ddgs import DDGS
from ddgs.exceptions import RatelimitException, DDGSException
import logging

# Set up logging
logging.getLogger("strands").setLevel(logging.INFO)

# Create a web search tool
@tool
def websearch(keywords: str, max_results: int = 3) -> str:
    """Search the web for information.
    Args:
        keywords (str): What to search for
        max_results (int): How many results to return
    Returns:
        Search results as text
    """
    try:
        results = DDGS().text(keywords, max_results=max_results)
        return results if results else "No results found."
    except Exception as e:
        return f"Search error: {e}"

print("Web search tool created successfully!")

Now create the recipe assistant agent that uses the web search tool to help with cooking questions.

In [ ]:

# Create the recipe assistant agent
recipe_agent = Agent(callback_handler=None,
    model="amazon.nova-lite-v1:0",
    system_prompt="""You are RecipeBot, a helpful cooking assistant.
    Help users find recipes and answer cooking questions.
    Use the websearch tool to find recipes and cooking information.""",
    tools=[websearch]
)

print("Recipe assistant agent created successfully!")

Test your recipe assistant by asking it for cooking help. Watch how it uses the web search tool to find current information.

In [ ]:
# Test the recipe assistant
response = recipe_agent("Suggest a simple recipe with chicken and broccoli.")
print(response)

You have successfully created a practical AI agent that can search the web and help with cooking questions. This demonstrates how agents can be useful for real-world tasks.

You have now experimented with the Strands Agents framework, which provides a way to build AI agents that can use tools and take actions. Using this framework, you have learned how agents differ from traditional LLMs by being able to actively use tools to accomplish tasks.

### Try it yourself
- Modify the system prompts to create agents for different use cases.
- Create custom tools for specific tasks your team might need.
- Experiment with different model configurations to understand how they affect agent behavior.

### Cleanup

You have completed this notebook. To move to the next part of the lab, do the following:

- Close this notebook file and continue with the **Conclusion**.